# 03 — References (Semantic Scholar)

Owner: Member 3. For every paper, asks the Semantic Scholar Graph API which works it cites
and stores them in `<paper-folder>/references.json`.

Functions provided to the rest of the pipeline (agreed interface):

| Function | Returns |
|---|---|
| `fetch_references(arxiv_id)` | `dict`: `id, status, n_refs, n_arxiv_refs, seconds, error`, and writes `references.json` |
| `submit_references(arxiv_id)` | a `Future` of the same dict, run in the background S2 thread |
| `load_references(arxiv_id)` | the saved `references.json`, or `None` |
| `crawl_references(ids)` | runs `fetch_references` over many IDs, skipping finished ones |
| `references_summary(ids)` | numbers for the report (success rate, references per paper, ...) |
| `references_rows(ids)` | rows of the `references` dataset table (arXiv-keyed references only) |

`status` is `ok` (paper found, its references were saved, possibly 0 of them), `not_found`
(Semantic Scholar does not index the paper yet) or `error` (the request kept failing).

### How it works

1. **One request per paper.** `GET /graph/v1/paper/arXiv:{id}` with the flat fields
   `references.title, references.authors, references.year, references.publicationDate,
   references.venue, references.externalIds` (plus `paperId`, which every reference returns).
   Semantic Scholar does not tell versions apart, so the references are stored once per paper,
   and they apply to every version.
2. **Keep everything, flag arXiv.** All references are kept locally, as the lab book asks;
   a reference with `externalIds.ArXiv` gets `arxiv_id` (as given, so old-style IDs such as
   `hep-th/9901001` stay unchanged) and `arxiv_folder_id` (`2310-12345`, the folder naming of
   `00_config`). Only those become rows of the `references` table at the final submission.
3. **Missing dates.** `submission_date` is the `publicationDate`; if S2 has none, the `year`;
   if that is missing too, the month encoded in the arXiv ID (`2310.12345` → `2023-10`).
   `date_source` records which one was used.

**Rate limits and errors.** One shared limiter keeps 1 request/s with an API key
(1 request every 3 s without one, to stay under 100 requests per 5 minutes). HTTP 429, 5xx
and network errors are retried with a growing wait (2 s, 4 s, 8 s, ... up to 60 s, or the
server's `Retry-After`); 404 means the paper is not indexed. Every call appends one row to
`logs/references.csv`, so the log says which paper failed and why.

**API key.** Read from the `S2_API_KEY` environment variable or Colab Secrets in
`00_config.ipynb`, never written in a notebook.

In [ ]:
if "CONFIG_LOADED" not in globals():
    get_ipython().run_line_magic("run", "-i ./00_config.ipynb")

import random
import threading
from concurrent.futures import ThreadPoolExecutor

import requests

## 1. HTTP settings and rate limiter

`S2_LIMITER` is shared by every caller and every thread, so the 1 request/s limit holds even
when references run in parallel with the arXiv downloads.

In [ ]:
S2_PAPER_URL = "https://api.semanticscholar.org/graph/v1/paper/arXiv:{}"
# Flat field names: nested forms such as references.authors.name are rejected by the API
S2_FIELDS = ",".join([
    "paperId", "title", "externalIds",
    "references.paperId", "references.title", "references.authors", "references.year",
    "references.publicationDate", "references.venue", "references.externalIds",
])
S2_RETRIES = 8           # attempts per paper before giving up (about 3 min of waiting)
S2_BACKOFF = 2.0         # first wait after a 429 / 5xx / network error, doubled each time
S2_MAX_BACKOFF = 60.0
S2_TIMEOUT = 30          # seconds per HTTP request

S2_SESSION = requests.Session()
S2_SESSION.headers["User-Agent"] = USER_AGENT
if S2_API_KEY:
    S2_SESSION.headers["x-api-key"] = S2_API_KEY
    S2_INTERVAL = S2_DELAY
else:
    # Unauthenticated pool: 100 requests per 5 minutes, i.e. one every 3 s
    S2_INTERVAL = max(S2_DELAY, 3.0)
    print("[references] S2_API_KEY not set: using the shared unauthenticated pool (slower, more 429s)")


class RateLimiter:
    """At most one call every `interval` seconds, shared by all threads."""

    def __init__(self, interval):
        self.interval = interval
        self._lock = threading.Lock()
        self._next = 0.0

    def wait(self):
        with self._lock:
            now = time.monotonic()
            if now < self._next:
                time.sleep(self._next - now)
            self._next = time.monotonic() + self.interval


S2_LIMITER = RateLimiter(S2_INTERVAL)

REFERENCE_STATS = {"requests": 0, "retries": 0, "rate_limited": 0, "seconds": 0.0}
_STATS_LOCK = threading.Lock()


def _count(key, amount=1):
    with _STATS_LOCK:
        REFERENCE_STATS[key] += amount

## 2. Request one paper

Only the errors that can go away on their own (429, 5xx, timeouts) are retried.
A 404 means S2 has no such paper, and other 4xx codes (bad key, bad request) will not
change on retry, so both stop at once.

In [ ]:
class S2Error(Exception):
    """A failed Semantic Scholar request; `attempts` is how many calls were made."""

    def __init__(self, message, attempts):
        super().__init__(message)
        self.attempts = attempts


class S2NotFound(S2Error):
    """Semantic Scholar has no paper with this arXiv ID."""


def _retry_after(response):
    """Seconds asked for by the server's Retry-After header, or None."""
    try:
        return float(response.headers.get("Retry-After"))
    except (TypeError, ValueError):
        return None


def _request_s2(arxiv_id):
    """JSON of one paper with its references; returns (data, attempts)."""
    url = S2_PAPER_URL.format(arxiv_id)
    wait = S2_BACKOFF
    last_error = None
    for attempt in range(1, S2_RETRIES + 1):
        S2_LIMITER.wait()
        _count("requests")
        server_wait = None
        try:
            response = S2_SESSION.get(url, params={"fields": S2_FIELDS}, timeout=S2_TIMEOUT)
        except requests.RequestException as err:
            last_error = f"network error ({err.__class__.__name__})"
        else:
            code = response.status_code
            if code == 200:
                return response.json(), attempt
            if code == 404:
                raise S2NotFound("paper not indexed by Semantic Scholar", attempt)
            if code == 429 or code >= 500:
                last_error = f"HTTP {code}"
                server_wait = _retry_after(response)
                if code == 429:
                    _count("rate_limited")
            else:
                hint = " (check S2_API_KEY)" if code in (401, 403) else ""
                raise S2Error(f"HTTP {code}{hint}: {response.text[:200]}", attempt)

        if attempt < S2_RETRIES:
            _count("retries")
            # A little random jitter so two restarted runs do not retry in lock-step
            time.sleep(max(wait, server_wait or 0) + random.uniform(0, 1))
            wait = min(wait * 2, S2_MAX_BACKOFF)
    raise S2Error(f"gave up after {S2_RETRIES} attempts, last: {last_error}", S2_RETRIES)

## 3. Turn the API result into reference records

S2 gives arXiv IDs without the `arXiv:` prefix and without a version, but both are stripped
anyway in case one slips through. A value that is not a valid new- or old-style ID is not
treated as an arXiv reference; it is still visible in `external_ids`.

In [ ]:
def normalize_ref_arxiv_id(raw):
    """S2 'ArXiv' value -> (arxiv_id, folder_id); (None, None) if it is not an arXiv ID.

    '2310.12345v2' -> ('2310.12345', '2310-12345'); 'hep-th/9901001' -> ('hep-th/9901001', 'hep-th_9901001').
    """
    if not raw:
        return None, None
    value = re.sub(r"^arxiv:", "", str(raw).strip(), flags=re.I)
    base, _ = split_version(value)
    if NEW_ID_RE.match(base) or OLD_ID_RE.match(base):
        return base, to_folder_id(base)
    return None, None


def _month_from_arxiv_id(arxiv_id):
    """'2310.12345' -> '2023-10', 'hep-th/9901001' -> '1999-01' (arXiv started in 1991)."""
    m = NEW_ID_RE.match(arxiv_id) or OLD_ID_RE.match(arxiv_id)
    yymm = m.group(1) if NEW_ID_RE.match(arxiv_id) else m.group(2)[:4]
    yy, mm = int(yymm[:2]), yymm[2:]
    return f"{1900 + yy if yy >= 91 else 2000 + yy}-{mm}"


def _ref_date(ref, arxiv_id):
    """Best available date of a reference and where it came from."""
    if ref.get("publicationDate"):
        return ref["publicationDate"], "publicationDate"
    if ref.get("year"):
        return str(ref["year"]), "year"
    if arxiv_id:
        return _month_from_arxiv_id(arxiv_id), "arxiv_id"
    return None, None


def _reference_record(ref):
    external_ids = ref.get("externalIds") or {}
    arxiv_id, folder_id = normalize_ref_arxiv_id(external_ids.get("ArXiv"))
    date, date_source = _ref_date(ref, arxiv_id)
    return {
        "s2_id": ref.get("paperId"),
        "title": ref.get("title"),
        "authors": [a["name"] for a in ref.get("authors") or [] if a.get("name")],
        "submission_date": date,
        "date_source": date_source,
        "year": ref.get("year"),
        "venue": ref.get("venue") or None,
        "arxiv_id": arxiv_id,
        "arxiv_folder_id": folder_id,
        "external_ids": external_ids,
    }

## 4. `fetch_references` and `references.json`

`references.json` is written for every paper, also when S2 does not know it (empty list), so
each paper folder is complete. An `error` result never overwrites a file that an earlier run
saved successfully.

Entries with neither an S2 ID nor a title carry no information (S2 returns these when a
publisher hides the reference list); they are dropped and counted in `n_dropped`.

In [ ]:
REF_LOG = LOG_DIR / "references.csv"
REF_LOG_FIELDS = ["id", "status", "s2_id", "n_refs", "n_arxiv_refs", "n_dropped",
                  "attempts", "seconds", "error", "logged_at"]
_REF_LOG_LOCK = threading.Lock()


def _log_reference(row):
    """Append one row to logs/references.csv (one row per fetch_references call)."""
    row = {**row, "logged_at": to_iso(datetime.now(timezone.utc))}
    with _REF_LOG_LOCK:
        new_file = not REF_LOG.exists()
        with open(REF_LOG, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=REF_LOG_FIELDS, extrasaction="ignore")
            if new_file:
                writer.writeheader()
            writer.writerow(row)


def references_path(arxiv_id):
    return DATA_DIR / to_folder_id(arxiv_id) / "references.json"


def load_references(arxiv_id):
    return read_json(references_path(arxiv_id))


def fetch_references(arxiv_id):
    """Crawl the references of one paper, save references.json and return a summary dict."""
    start = time.perf_counter()
    base, _ = split_version(arxiv_id)
    doc = {
        "id": base, "status": None, "s2_id": None, "s2_title": None,
        "n_refs": 0, "n_arxiv_refs": 0, "n_dropped": 0, "error": None,
        "source": "Semantic Scholar Graph API (one list per paper, shared by all versions)",
        "references": [],
    }
    attempts = 0
    try:
        data, attempts = _request_s2(base)
    except S2NotFound as err:
        doc.update(status="not_found", error=str(err))
        attempts = err.attempts
    except Exception as err:  # keep the pipeline going; the reason goes to the log
        doc.update(status="error", error=f"{err.__class__.__name__}: {err}")
        attempts = getattr(err, "attempts", attempts)
    else:
        raw = data.get("references") or []
        kept = [r for r in raw if r.get("paperId") or r.get("title")]
        refs = [_reference_record(r) for r in kept]
        doc.update(
            status="ok", s2_id=data.get("paperId"), s2_title=data.get("title"),
            n_refs=len(refs), n_arxiv_refs=sum(r["arxiv_id"] is not None for r in refs),
            n_dropped=len(raw) - len(kept), references=refs,
        )
    doc["fetched_at"] = to_iso(datetime.now(timezone.utc))

    previous = load_references(base)
    if not (doc["status"] == "error" and previous and previous.get("status") == "ok"):
        write_json(paper_dir(base) / "references.json", doc)

    seconds = time.perf_counter() - start
    _count("seconds", seconds)
    summary = {key: doc[key] for key in ("id", "status", "n_refs", "n_arxiv_refs", "error")}
    summary["seconds"] = round(seconds, 3)
    _log_reference({**summary, "s2_id": doc["s2_id"], "n_dropped": doc["n_dropped"],
                    "attempts": attempts})
    return summary

## 5. Running in parallel with the downloads

The arXiv limit (3 s) and the Semantic Scholar limit (1 s) are separate, so references can be
fetched while `02_download` waits for arXiv. `submit_references` runs `fetch_references` on a
single background thread: still one S2 connection, still 1 request/s. Use in `05_run_pipeline`:

```python
future = submit_references(pid)          # starts at once in the S2 thread
for v in range(1, n_versions + 1):
    download_version(pid, v)              # arXiv, 3 s apart, in the main thread
refs = future.result()                    # usually already finished
```

Submitting the *next* paper before downloading the current one keeps the S2 thread busy
all the time. `crawl_references` is for a references-only pass, for example to retry the
papers that ended in `error`.

In [ ]:
_REF_EXECUTOR = None


def submit_references(arxiv_id):
    """fetch_references(arxiv_id) in the background S2 thread; returns a Future."""
    global _REF_EXECUTOR
    if _REF_EXECUTOR is None:
        _REF_EXECUTOR = ThreadPoolExecutor(max_workers=1, thread_name_prefix="s2")
    return _REF_EXECUTOR.submit(fetch_references, arxiv_id)


def crawl_references(ids, redo_errors=True, verbose=True):
    """fetch_references for every ID, skipping papers whose references.json is final."""
    final = ("ok", "not_found") if redo_errors else ("ok", "not_found", "error")
    results = []
    for i, arxiv_id in enumerate(ids, 1):
        saved = load_references(arxiv_id)
        if saved and saved.get("status") in final:
            continue
        result = fetch_references(arxiv_id)
        results.append(result)
        if verbose:
            print(f"[{i}/{len(ids)}] {result['id']:<16} {result['status']:<10} "
                  f"refs={result['n_refs']:<4} arxiv={result['n_arxiv_refs']:<4} "
                  f"{result['seconds']:.1f}s {result['error'] or ''}")
    return results

## 6. Numbers for the report and the `references` table

`references_summary` reads the saved files, so it gives the same numbers whether the crawl ran
in one go or over several restarts. The report needs the average number of references per paper
and the success rate of reference scraping:

- **success rate**: papers with `status == "ok"` / papers attempted;
- **references per paper**: averaged over the successful papers;
- **arXiv share**: references with an arXiv ID / all references.

`references_rows` gives one row per arXiv-keyed reference in the column format of the
`references` table (Milestone 1 statement), ready for the final Hugging Face dataset.

In [ ]:
def _saved_references(ids=None):
    """references.json of the given IDs, or of every paper folder in DATA_DIR."""
    if ids is None:
        paths = sorted(DATA_DIR.glob("*/references.json"))
    else:
        paths = [references_path(i) for i in ids]
    return [doc for doc in (read_json(p) for p in paths) if doc]


def references_summary(ids=None):
    docs = _saved_references(ids)
    ok = [d for d in docs if d["status"] == "ok"]
    refs = [r for d in ok for r in d["references"]]
    n_arxiv = sum(r["arxiv_id"] is not None for r in refs)

    def share(part, whole):
        return round(part / whole, 4) if whole else None

    return {
        "papers": len(docs),
        "ok": len(ok),
        "not_found": sum(d["status"] == "not_found" for d in docs),
        "error": sum(d["status"] == "error" for d in docs),
        "success_rate": share(len(ok), len(docs)),
        "papers_with_refs": sum(d["n_refs"] > 0 for d in ok),
        "refs_total": len(refs),
        "arxiv_refs_total": n_arxiv,
        "avg_refs_per_paper": share(len(refs), len(ok)),
        "avg_arxiv_refs_per_paper": share(n_arxiv, len(ok)),
        "arxiv_share": share(n_arxiv, len(refs)),
        "old_style_arxiv_refs": sum(r["arxiv_id"] is not None and "/" in r["arxiv_id"] for r in refs),
        "refs_without_date": sum(r["submission_date"] is None for r in refs),
        "date_sources": {src: sum(r["date_source"] == src for r in refs)
                         for src in ("publicationDate", "year", "arxiv_id", None)},
    }


def references_rows(ids=None):
    """Rows of the `references` table: only references that carry an arXiv ID."""
    return [
        {
            "arxiv_id": d["id"],
            "ref_arxiv_id": r["arxiv_id"],
            "ref_title": r["title"],
            "ref_authors": r["authors"],
            "ref_submission_date": r["submission_date"],
            "ref_s2_id": r["s2_id"],
        }
        for d in _saved_references(ids) if d["status"] == "ok"
        for r in d["references"] if r["arxiv_id"]
    ]

## 7. Quick test

Runs only when this notebook is opened on its own (`RUN_TEST = True`). The sample covers a
heavily cited paper (`1706.03762`), an old-style ID (`hep-th/9711200`), a recent paper,
an ID that does not exist (`2310.99999`) and the first paper of our assigned range.

In [ ]:
if RUN_TEST:
    # Offline checks of the ID and date helpers
    assert normalize_ref_arxiv_id("2310.12345") == ("2310.12345", "2310-12345")
    assert normalize_ref_arxiv_id("arXiv:2310.12345v2") == ("2310.12345", "2310-12345")
    assert normalize_ref_arxiv_id("1401.0001") == ("1401.0001", "1401-0001")
    assert normalize_ref_arxiv_id("hep-th/9901001") == ("hep-th/9901001", "hep-th_9901001")
    assert normalize_ref_arxiv_id("math.AG/0101001") == ("math.AG/0101001", "math-AG_0101001")
    assert normalize_ref_arxiv_id("not an id") == (None, None)
    assert normalize_ref_arxiv_id(None) == (None, None)
    assert _ref_date({"publicationDate": "2017-06-12", "year": 2017}, None) == ("2017-06-12", "publicationDate")
    assert _ref_date({"year": 2017}, None) == ("2017", "year")
    assert _ref_date({}, "hep-th/9901001") == ("1999-01", "arxiv_id")
    assert _ref_date({}, "2310.12345") == ("2023-10", "arxiv_id")
    assert _ref_date({}, None) == (None, None)

    TEST_IDS = ["1706.03762", "hep-th/9711200", "2310.06825", "2310.99999", ID_RANGES[0][0]]
    crawl_references(TEST_IDS, verbose=True)
    print(json.dumps(references_summary(TEST_IDS), indent=2))
    print(REFERENCE_STATS)
    print(references_rows(TEST_IDS)[:2])